# Baseline

### Definitions: 
**sii** - Severity Impairment Index - target \
0 = None\
1 = Mild\
2 = Moderate\
3 = Severe


**QWK** - Quadratic Weighted Kappa - measures agreement while penalizing more distant class errors more strongly


**PCIAT** — Parent-Child Internet Addiction Test - a 20-question questionnaire with answers from 0 to 5. `PCIAT-PCIAT_Total` is the sum of those answers.\
According to local dictionary:\
0–30 → sii = 0\
31–49 → sii = 1\
50–79 → sii = 2\
80–100 → sii = 3\
All `PCIAT-*` columns are excluded from model features because they directly reveal information used to calculate the target. Including them would cause target leakage.


**CGAS** - Children's Global Assessment Scale - Numeric scale used by mental health clinicians to rate the general functioning of youths under the age of 18




## Minimal data audit

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()

# go to parent folder if you are currently in notebooks
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Make shared project modules importable from the notebooks directory
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ID_COLUMN, LEAKAGE_PREFIX, RANDOM_STATE, TARGET

DATA_DIR = PROJECT_ROOT / "data" / "child-mind-institute-problematic-internet-use"
TRAIN_PATH = DATA_DIR / "train.csv"

train = pd.read_csv(TRAIN_PATH)

print("Train path:", TRAIN_PATH.relative_to(PROJECT_ROOT))
print("Train shape:", train.shape)
print("Target present:", TARGET in train.columns)
print("Rows with missing target:", train[TARGET].isna().sum())

print("\nTarget distribution:")
print(train[TARGET].value_counts(dropna=False).sort_index())

Train path: data/child-mind-institute-problematic-internet-use/train.csv
Train shape: (3960, 82)
Target present: True
Rows with missing target: 1224

Target distribution:
sii
0.0    1594
1.0     730
2.0     378
3.0      34
NaN    1224
Name: count, dtype: int64


In [2]:
# Keep only rows available for supervised training
labeled_train = train.dropna(subset=[TARGET]).copy()

categorical_columns = train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_columns = train.select_dtypes(
    include="number"
).columns.drop(TARGET).tolist()

print("Labelled shape:", labeled_train.shape)
print("Number of categorical columns:", len(categorical_columns))
print("Categorical columns:")
print(categorical_columns)

print("\nNumber of numerical feature columns:", len(numerical_columns))
print("Numerical columns:")
print(numerical_columns)

print("\nAll column dtypes:")
print(train.dtypes.to_string())

Labelled shape: (2736, 82)
Number of categorical columns: 12
Categorical columns:
['id', 'Basic_Demos-Enroll_Season', 'CGAS-Season', 'Physical-Season', 'Fitness_Endurance-Season', 'FGC-Season', 'BIA-Season', 'PAQ_A-Season', 'PAQ_C-Season', 'PCIAT-Season', 'SDS-Season', 'PreInt_EduHx-Season']

Number of numerical feature columns: 69
Numerical columns:
['Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score', 'Physical-BMI', 'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference', 'Physical-Diastolic_BP', 'Physical-HeartRate', 'Physical-Systolic_BP', 'Fitness_Endurance-Max_Stage', 'Fitness_Endurance-Time_Mins', 'Fitness_Endurance-Time_Sec', 'FGC-FGC_CU', 'FGC-FGC_CU_Zone', 'FGC-FGC_GSND', 'FGC-FGC_GSND_Zone', 'FGC-FGC_GSD', 'FGC-FGC_GSD_Zone', 'FGC-FGC_PU', 'FGC-FGC_PU_Zone', 'FGC-FGC_SRL', 'FGC-FGC_SRL_Zone', 'FGC-FGC_SRR', 'FGC-FGC_SRR_Zone', 'FGC-FGC_TL', 'FGC-FGC_TL_Zone', 'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMI', 'BIA-BIA_BMR', 'BIA-BIA_DEE', 'BIA-BIA_

In [3]:
import numpy as np

# Missing values in the complete train table
missing_summary = (
    train.isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "missing_count", "mean": "missing_percent"})
)

missing_summary["missing_percent"] *= 100
missing_summary = missing_summary.query("missing_count > 0")
missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False,
)

print("Columns containing missing values:", len(missing_summary))
print("\nTop 20 columns by missing percentage:")
print(missing_summary.head(20).round(2).to_string())

# Duplicates and ID checks
print("\nExact duplicate rows:", train.duplicated().sum())
print("Missing IDs:", train["id"].isna().sum())
print("Unique IDs:", train["id"].nunique())
print("Duplicated IDs:", train["id"].duplicated().sum())

# Potential leakage columns
pciat_columns = [
    column for column in train.columns
    if column.startswith("PCIAT-")
]

print("\nPCIAT columns:", len(pciat_columns))
print(pciat_columns)

# Check whether sii is directly determined by PCIAT total
known_pciat = train[
    train["sii"].notna() & train["PCIAT-PCIAT_Total"].notna()
].copy()

derived_sii = pd.cut(
    known_pciat["PCIAT-PCIAT_Total"],
    bins=[-np.inf, 30, 49, 79, np.inf],
    labels=[0, 1, 2, 3],
).astype(int)

print("\nRows with both sii and PCIAT total:", len(known_pciat))
print(
    "Rows where derived class differs from sii:",
    (derived_sii.to_numpy() != known_pciat["sii"].to_numpy()).sum(),
)

Columns containing missing values: 78

Top 20 columns by missing percentage:
                              missing_count  missing_percent
PAQ_A-Season                         3485.0            88.01
PAQ_A-PAQ_A_Total                    3485.0            88.01
Fitness_Endurance-Time_Sec           3220.0            81.31
Fitness_Endurance-Time_Mins          3220.0            81.31
Fitness_Endurance-Max_Stage          3217.0            81.24
Physical-Waist_Circumference         3062.0            77.32
FGC-FGC_GSND_Zone                    2898.0            73.18
FGC-FGC_GSD_Zone                     2897.0            73.16
FGC-FGC_GSD                          2886.0            72.88
FGC-FGC_GSND                         2886.0            72.88
Fitness_Endurance-Season             2652.0            66.97
PAQ_C-PAQ_C_Total                    2239.0            56.54
PAQ_C-Season                         2239.0            56.54
BIA-BIA_FFMI                         1969.0            49.72
BIA-BIA_

### Key Findings

- The training dataset contains **3,960 rows and 82 columns**
- Only **2,736 rows have a known target**; 1,224 rows with missing `sii` must be excluded from supervised training and cross-validation.
- The target is highly imbalanced:
  - Class 0: 1,594 observations
  - Class 1: 730 observations
  - Class 2: 378 observations
  - Class 3: 34 observations
- 12 string and 69 numerical feature columns.
- Missing values are widespread: 78 columns contain missing values in the complete training table.
- No exact duplicate rows.
- The `id` column is an identifier
- `sii` is directly derived from `PCIAT-PCIAT_Total`: all 2,736 labelled rows match the documented PCIAT score thresholds

## Dataset Preparation and Cross-Validation Setup

In [4]:
from src.evaluation import (
    create_cv_splits,
    evaluate_model,
    quadratic_weighted_kappa,
    regression_to_classes,
)

# Define columns excluded from model features
excluded_columns = {
    TARGET,
    ID_COLUMN,
    *[
        column
        for column in labeled_train.columns
        if column.startswith(LEAKAGE_PREFIX)
    ],
}

feature_columns = [
    column
    for column in labeled_train.columns
    if column not in excluded_columns
]

X = labeled_train[feature_columns].copy()

# sii was loaded as float because it contains missing values in the original table
# now it is safe to convert it to integer
y = labeled_train[TARGET].astype(int)

cv_splits = create_cv_splits(
    X=X,
    y=y,
    ids=labeled_train[ID_COLUMN],  # Keep participant folds stable after row reordering
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Excluded columns:", len(excluded_columns))
print("Remaining features:", len(feature_columns))

for fold_number, (train_indices, validation_indices) in enumerate(
    cv_splits,
    start=1,
):
    fold_distribution = (
        y.iloc[validation_indices]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    print(
        f"Fold {fold_number}: "
        f"train={len(train_indices)}, "
        f"validation={len(validation_indices)}, "
        f"validation classes={fold_distribution}"
    )

X shape: (2736, 58)
y shape: (2736,)
Excluded columns: 24
Remaining features: 58
Fold 1: train=2188, validation=548, validation classes={0: 319, 1: 146, 2: 76, 3: 7}
Fold 2: train=2189, validation=547, validation classes={0: 319, 1: 146, 2: 76, 3: 6}
Fold 3: train=2189, validation=547, validation classes={0: 318, 1: 146, 2: 76, 3: 7}
Fold 4: train=2189, validation=547, validation classes={0: 319, 1: 146, 2: 75, 3: 7}
Fold 5: train=2189, validation=547, validation classes={0: 319, 1: 146, 2: 75, 3: 7}


## Evaluation Metric

In [5]:
print("Perfect QWK:", quadratic_weighted_kappa(y, y))

Perfect QWK: 1.0


## Preprocessing

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include="number"
).columns.tolist()

numerical_pipeline = Pipeline(
    steps=[
        # Median is less sensitive to extreme values than the mean
        ("imputer", SimpleImputer(strategy="median")),
        # Ridge regularization depends on feature scale
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        # Use the most common training-fold category for missing values
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                # A validation fold may contain a category absent from its training fold
                handle_unknown="ignore",
            ),
        ),
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    # Only explicitly listed model features are transformed
    remainder="drop",
)

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features before encoding:", len(X.columns))

Numerical features: 48
Categorical features: 10
Total features before encoding: 58


## DummyClassifier Baseline

In [7]:
from sklearn.dummy import DummyClassifier


dummy_model = DummyClassifier(
    strategy="most_frequent",  # Predict the majority class from each training fold
)

dummy_result = evaluate_model(
    model=dummy_model,
    X=X,
    y=y,
    cv_splits=cv_splits,
)

Fold 1: training QWK=0.0000, validation QWK=0.0000
Fold 2: training QWK=0.0000, validation QWK=0.0000
Fold 3: training QWK=0.0000, validation QWK=0.0000
Fold 4: training QWK=0.0000, validation QWK=0.0000
Fold 5: training QWK=0.0000, validation QWK=0.0000
Mean training QWK: 0.0000
Mean validation QWK: 0.0000
Validation QWK standard deviation: 0.0000


## Ridge Regression Baseline

In [8]:
from sklearn.linear_model import Ridge

In [9]:
ridge_model = Pipeline(
    steps=[
        # Fit imputation, scaling, and encoding separately in each fold
        ("preprocessor", preprocessor),
        (
            "model",
            Ridge(
                alpha=1.0,  # Start with sklearn's default regularization strength
            ),
        ),
    ]
)

ridge_result = evaluate_model(
    model=ridge_model,
    X=X,
    y=y,
    cv_splits=cv_splits,
    # Convert continuous Ridge predictions to classes before QWK
    prediction_transform=regression_to_classes,
)

Fold 1: training QWK=0.4031, validation QWK=0.3323
Fold 2: training QWK=0.3910, validation QWK=0.4424
Fold 3: training QWK=0.4009, validation QWK=0.3602
Fold 4: training QWK=0.4102, validation QWK=0.3469
Fold 5: training QWK=0.4154, validation QWK=0.3564
Mean training QWK: 0.4041
Mean validation QWK: 0.3677
Validation QWK standard deviation: 0.0386


### Ridge Coefficient Sanity Check

In [10]:
# Fit once on all labelled data for coefficient inspection
ridge_model.fit(X, y)

fitted_preprocessor = ridge_model.named_steps["preprocessor"]
fitted_ridge = ridge_model.named_steps["model"]

transformed_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

ridge_coefficients = pd.DataFrame(
    {
        "feature": transformed_feature_names,
        "coefficient": fitted_ridge.coef_,
    }
)

ridge_coefficients["absolute_coefficient"] = (
    ridge_coefficients["coefficient"].abs()
)

ridge_coefficients = ridge_coefficients.sort_values(
    "absolute_coefficient",
    ascending=False,
)

print("Intercept:", fitted_ridge.intercept_)
print("Number of coefficients:", len(ridge_coefficients))

ridge_coefficients.head(15)

Intercept: 0.5474257197371591
Number of coefficients: 88


,feature,coefficient,absolute_coefficient
36,numerical__BIA-BIA_Fat,0.387102,0.387102
38,numerical__BIA-BIA_ICW,-0.356881,0.356881
32,numerical__BIA-BIA_ECW,0.331280,0.331280
46,numerical__SDS-SDS_Total_T,0.270412,0.270412
28,numerical__BIA-BIA_BMC,0.269634,0.269634
40,numerical__BIA-BIA_LST,-0.261525,0.261525
31,numerical__BIA-BIA_DEE,0.247863,0.247863
0,numerical__Basic_Demos-Age,0.158762,0.158762
41,numerical__BIA-BIA_SMM,0.155557,0.155557
74,categorical__PAQ_A-Season_Summer,-0.154603,0.154603


## Decision Tree Baseline

In [11]:
from sklearn.tree import DecisionTreeClassifier

In [12]:
decision_tree_model = Pipeline(
    steps=[
        # Fit preprocessing separately inside each fold
        ("preprocessor", preprocessor),
        (
            "model",
            DecisionTreeClassifier(
                random_state=RANDOM_STATE,  # Reproduce tree tie-breaking
            ),
        ),
    ]
)

decision_tree_result = evaluate_model(
    model=decision_tree_model,
    X=X,
    y=y,
    cv_splits=cv_splits,
)

Fold 1: training QWK=1.0000, validation QWK=0.1830
Fold 2: training QWK=1.0000, validation QWK=0.2447
Fold 3: training QWK=1.0000, validation QWK=0.3019
Fold 4: training QWK=1.0000, validation QWK=0.2140
Fold 5: training QWK=1.0000, validation QWK=0.1610
Mean training QWK: 1.0000
Mean validation QWK: 0.2209
Validation QWK standard deviation: 0.0494


## Random Forest Baseline

In [13]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
random_forest_model = Pipeline(
    steps=[
        # Fit preprocessing separately inside each fold
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,  # Average enough trees for a stable baseline
                random_state=RANDOM_STATE,  # Reproduce tree sampling and tie-breaking
                n_jobs=-1,  # Use all available CPU cores
            ),
        ),
    ]
)

random_forest_result = evaluate_model(
    model=random_forest_model,
    X=X,
    y=y,
    cv_splits=cv_splits,
)

## Baseline Results

In [ ]:
results_by_model = {
    "DummyClassifier": dummy_result,
    "Ridge": ridge_result,
    "DecisionTreeClassifier": decision_tree_result,
    "RandomForestClassifier": random_forest_result,
}

result_rows = []

for model_name, model_result in results_by_model.items():
    training_scores = model_result["training_scores"]
    validation_scores = model_result["validation_scores"]
    oof_counts = np.bincount(
        model_result["oof_predictions"],
        minlength=4,  # Include classes with zero predictions
    )

    row = {
        "model": model_name,
        "mean_training_qwk": np.mean(training_scores),
        "mean_validation_qwk": np.mean(validation_scores),
        "validation_std_qwk": np.std(validation_scores),
        "oof_pred_0_count": oof_counts[0],
        "oof_pred_1_count": oof_counts[1],
        "oof_pred_2_count": oof_counts[2],
        "oof_pred_3_count": oof_counts[3],
    }

    row.update(
        {
            f"fold_{fold_number}_qwk": score
            for fold_number, score in enumerate(validation_scores, start=1)
        }
    )
    result_rows.append(row)

baseline_results = pd.DataFrame(result_rows)
baseline_results = baseline_results[
    [
        "model",
        "fold_1_qwk",
        "fold_2_qwk",
        "fold_3_qwk",
        "fold_4_qwk",
        "fold_5_qwk",
        "mean_training_qwk",
        "mean_validation_qwk",
        "validation_std_qwk",
        "oof_pred_0_count",
        "oof_pred_1_count",
        "oof_pred_2_count",
        "oof_pred_3_count",
    ]
].round(4)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)  # Reuse the directory on later runs
RESULTS_PATH = RESULTS_DIR / "baseline_cv_results.csv"

baseline_results.to_csv(
    RESULTS_PATH,
    index=False,  # Keep the pandas row index out of the CSV
)

print("Saved results to:", RESULTS_PATH)
baseline_results

Saved results to: /home/arina/Desktop/predict-internet-usage-ivanov-secret/results/baseline_cv_results.csv


,model,fold_1_qwk,fold_2_qwk,fold_3_qwk,fold_4_qwk,fold_5_qwk,mean_qwk,std_qwk
0,DummyClassifier,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,Ridge,0.3323,0.4424,0.3602,0.3469,0.3564,0.3677,0.0386
2,DecisionTreeClassifier,0.1830,0.2447,0.3019,0.2140,0.1610,0.2209,0.0494
3,RandomForestClassifier,0.2944,0.3166,0.2919,0.3114,0.2461,0.2921,0.0249


## Baseline Summary

- Ridge achieved the best mean validation QWK: **0.3677**
- Ridge showed the smallest train-validation gap: **0.4041 vs 0.3677**
- Decision Tree overfit strongly: **1.0000 training QWK vs 0.2209 validation QWK**
- Random Forest also overfit: **1.0000 training QWK vs 0.2921 validation QWK**
- Ridge benefits from the ordered target because errors between nearby classes are smaller than errors between distant classes
- Ridge predicted mostly classes 0 and 1 and rarely predicted classes 2 or 3, so severe cases remain a baseline limitation
- No test data or leaderboard feedback was used